In [11]:
# imports
import pandas as pd
import torch
from torch import nn

%load_ext autoreload
%autoreload 2
import models as m
import evaluation as ev

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [13]:
fold_labels = pd.read_csv('data/fold_labels.csv')
img_parquet  = pd.read_parquet('data/clip_image.parquet')
txt_parquet  = pd.read_parquet('data/clip_text.parquet')

label_keys = set(fold_labels['external_code'])
missing_img = label_keys - set(img_parquet['external_code'])
missing_txt = label_keys - set(txt_parquet['external_code'])

# merge should have 5080 products
print("Fold labels shape:", fold_labels.shape)
print("Img parquet shape:", img_parquet.shape)
print("Text parquet shape:", txt_parquet.shape)
print("Missing img?", len(missing_img))
print("Missing txt?", len(missing_txt))
print("External code unique?", fold_labels['external_code'].nunique())


Fold labels shape: (20341, 5)
Img parquet shape: (5577, 769)
Text parquet shape: (5577, 769)
Missing img? 0
Missing txt? 0
External code unique? 5080


In [14]:
# index both towers by external code
txt_embeddings = txt_parquet.set_index('external_code')
img_embeddings = img_parquet.set_index('external_code')

is_fold_0_train = (fold_labels['fold'] == 0) & (fold_labels['split'] == 'train')
fold_0_train = fold_labels[is_fold_0_train]

train_codes = fold_0_train['external_code'].to_numpy()

train_txt_x = txt_embeddings.loc[train_codes]
train_img_x = img_embeddings.loc[train_codes]
train_labels = fold_0_train['label'].to_numpy()

print("Fold 0 shape:", fold_0_train.shape)
print("Text X shape", train_txt_x.shape)
print("Image X shape", train_img_x.shape)
print("Labels Length:", len(train_labels))

print((train_txt_x.index.to_numpy() == train_codes).all())
print((train_img_x.index.to_numpy() == train_codes).all())

Fold 0 shape: (3053, 5)
Text X shape (3053, 768)
Image X shape (3053, 768)
Labels Length: 3053
True
True


In [15]:
# index both towers by external code
is_fold_0_val = (fold_labels['fold'] == 0) & (fold_labels['split'] == 'val')
fold_0_val = fold_labels[is_fold_0_val]

val_codes = fold_0_val['external_code'].to_numpy()

train_labels = fold_0_val['label'].to_numpy()
in_buffer = fold_0_val['in_buffer'].to_numpy()

print("Non buffer:", (~in_buffer).sum())
print(fold_0_val['label'].value_counts().sort_index())

Non buffer: 769
label
0    253
1    509
2    254
Name: count, dtype: int64


In [16]:
# return fold's data for one split, aligned by product
def get_fold_data(fold, split):
    selected_rows = (fold_labels['fold'] == fold) & (fold_labels['split'] == split)
    fold_rows = fold_labels[selected_rows]

    codes = fold_rows['external_code'].to_numpy()

    fold_data = {
        'text': txt_embeddings.loc[codes],
        'image': img_embeddings.loc[codes],
        'labels': fold_rows['label'].to_numpy(),
        'in_buffer': fold_rows['in_buffer'].to_numpy(),
        'codes': codes,
    }

    return fold_data

print(get_fold_data(3, 'val')['text'].shape[0])
print((~get_fold_data(3, 'val')['in_buffer']).sum())

1016
772


In [17]:
# function to turn the fold data into tensors for pytorch

def to_tensors(fold_data, device):
    text_tensor = torch.from_numpy(fold_data['text'].values).to(device)
    image_tensor = torch.from_numpy(fold_data['image'].values).to(device)

    labels_tensor = torch.as_tensor(fold_data['labels'], dtype=torch.int64).to(device)

    in_buffer = fold_data['in_buffer']
    codes = fold_data['codes']

    return {
        'text': text_tensor,
        'image': image_tensor,
        'labels': labels_tensor,
        'in_buffer': in_buffer,
        'codes': codes
    }

train_data = to_tensors(get_fold_data(0, 'train'), device)
val_data = to_tensors(get_fold_data(0, 'val'), device)

print(train_data['text'].shape, train_data['text'].dtype, train_data['text'].device)
print(train_data['image'].shape, train_data['image'].dtype, train_data['image'].device)
print(train_data['labels'].shape, train_data['labels'].dtype, train_data['labels'].device)
print(train_data['in_buffer'].dtype, len(train_data['in_buffer']), train_data['in_buffer'].sum())
print(train_data['codes'].dtype, len(train_data['codes']))


torch.Size([3053, 768]) torch.float32 mps:0
torch.Size([3053, 768]) torch.float32 mps:0
torch.Size([3053]) torch.int64 mps:0
bool 3053 0
int64 3053


In [18]:
dropout = 0.2
n_classes = 3
batch_size = 64
learning_rate = 1e-3
max_epochs = 50
seed = 7

def train_one_run (condition, hidden_width, fold, train_data, val_data):
    torch.manual_seed(seed)

    epoch_rows = []
    input_dim = train_data['text'].shape[1]

    config = {
        'condition': condition,
        'input_dim': input_dim,
        'hidden_width': hidden_width,
        'dropout': dropout,
        'n_classes': n_classes,
    }

    model = m.ConditionModel(config).to(train_data['text'].device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    loss_function = nn.CrossEntropyLoss()
    n_train = train_data['text'].shape[0]

    for epoch in range(1, max_epochs +1):
        model.train()
        running_loss = 0.0

        shuffled = torch.randperm(n_train, device=train_data['text'].device)

        for start in range(0, n_train, batch_size):
            batch_rows = shuffled[start:start + batch_size]

            batch_text = train_data['text'][batch_rows]
            batch_image = train_data['image'][batch_rows]
            batch_labels = train_data['labels'][batch_rows]

            optimizer.zero_grad()
            logits = model(batch_text, batch_image)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(batch_rows)

        train_loss = running_loss / n_train

        model.eval()
        with torch.no_grad():
            val_logits = model(val_data['text'], val_data['image'])

        predictions = val_logits.argmax(dim=1).cpu().numpy()

        scores = ev.score_fold(
            val_data['labels'].cpu().numpy(),
            predictions,
            val_data['in_buffer'],
        )

        epoch_rows.append({
            'condition': condition,
            'hidden_width': hidden_width,
            'fold': fold,
            'epoch': epoch,
            'train_loss': train_loss,
            'balanced_accuracy_full': scores['balanced_accuracy_full']
        })

    return epoch_rows

In [19]:
check_model = m.ConditionModel({
    'condition': 'text',
    'input_dim': 768,
    'hidden_width': 256,
    'dropout': 0.2,
    'n_classes': 3
})

sum(p.numel() for p in check_model.parameters())

263427

In [22]:
fold_0_train = to_tensors(get_fold_data(0, 'train'), device)
fold_0_val = to_tensors(get_fold_data(0, 'val'), device)

text_rows = train_one_run('text', 256, 0, fold_0_train, fold_0_val)

curve = pd.DataFrame(text_rows)
display(curve.head(10))
display(curve.tail(5))

,condition,hidden_width,fold,epoch,train_loss,balanced_accuracy_full
0,text,256,0,1,1.079355,0.440056
1,text,256,0,2,1.047722,0.482243
2,text,256,0,3,1.026732,0.459421
3,text,256,0,4,1.003055,0.441089
4,text,256,0,5,0.997993,0.474754
5,text,256,0,6,0.981464,0.476181
6,text,256,0,7,0.972841,0.461623
7,text,256,0,8,0.958435,0.461732
8,text,256,0,9,0.941707,0.462203
9,text,256,0,10,0.934988,0.468956


,condition,hidden_width,fold,epoch,train_loss,balanced_accuracy_full
45,text,256,0,46,0.718691,0.457606
46,text,256,0,47,0.714355,0.465617
47,text,256,0,48,0.716713,0.445823
48,text,256,0,49,0.713182,0.453695
49,text,256,0,50,0.718292,0.465560


In [23]:
conditions = ['text', 'image', 'early', 'intermediate', 'late', 'learned']
widths = [128, 256, 512]
all_rows = []


# each fold runs once with each of the six conditions
for fold in range(5):
    train_data = to_tensors(get_fold_data(fold, 'train'), device)
    val_data = to_tensors(get_fold_data(fold, 'val'), device)

    for condition in conditions:
        for hidden_width in widths:
            print(f'fold {fold} | {condition} | {hidden_width}', flush=True)
            all_rows.extend(train_one_run(condition, hidden_width, fold, train_data, val_data))

results = pd.DataFrame(all_rows)
results.to_csv('data/cv_curves.csv', index=False)

# check that the loop is handing each run the right fold's data
# each fold's value should be distinct
subset = results[(results.condition == 'text') & (results.hidden_width == 256) & (results.epoch == 1)]
print(subset[['fold', 'train_loss', 'balanced_accuracy_full']])

fold 0 | text | 128
fold 0 | text | 256
fold 0 | text | 512
fold 0 | image | 128
fold 0 | image | 256
fold 0 | image | 512
fold 0 | early | 128
fold 0 | early | 256
fold 0 | early | 512
fold 0 | intermediate | 128
fold 0 | intermediate | 256
fold 0 | intermediate | 512
fold 0 | late | 128
fold 0 | late | 256
fold 0 | late | 512
fold 0 | learned | 128
fold 0 | learned | 256
fold 0 | learned | 512
fold 1 | text | 128
fold 1 | text | 256
fold 1 | text | 512
fold 1 | image | 128
fold 1 | image | 256
fold 1 | image | 512
fold 1 | early | 128
fold 1 | early | 256
fold 1 | early | 512
fold 1 | intermediate | 128
fold 1 | intermediate | 256
fold 1 | intermediate | 512
fold 1 | late | 128
fold 1 | late | 256
fold 1 | late | 512
fold 1 | learned | 128
fold 1 | learned | 256
fold 1 | learned | 512
fold 2 | text | 128
fold 2 | text | 256
fold 2 | text | 512
fold 2 | image | 128
fold 2 | image | 256
fold 2 | image | 512
fold 2 | early | 128
fold 2 | early | 256
fold 2 | early | 512
fold 2 | interme

In [24]:
# get the average for the six conditions at each width
averaged = (results
            .groupby(['condition', 'hidden_width', 'epoch'])['balanced_accuracy_full']
            .mean()
            .reset_index()
            )

peaks = averaged.loc[averaged.groupby(['condition', 'hidden_width'])['balanced_accuracy_full'].idxmax()]
print(peaks.sort_values('balanced_accuracy_full', ascending=False))

        condition  hidden_width  epoch  balanced_accuracy_full
479          late           128     30                0.536504
448  intermediate           512     49                0.533758
666       learned           256     17                0.532841
319  intermediate           128     20                0.532046
516          late           256     17                0.531006
374  intermediate           256     25                0.528740
638       learned           128     39                0.527705
30          early           128     31                0.527408
737       learned           512     38                0.527385
111         early           512     12                0.526938
599          late           512     50                0.526622
241         image           256     42                0.524503
69          early           256     20                0.523491
299         image           512     50                0.513614
163         image           128     14                0

In [25]:
for width in [128, 256, 512]:
    subset = averaged[(averaged.condition == 'intermediate') & (averaged.hidden_width == width)]
    print(width, subset.balanced_accuracy_full.iloc[-10:].values.round(4))

128 [0.5178 0.5215 0.5207 0.5165 0.5177 0.5099 0.5118 0.5066 0.5122 0.5095]
256 [0.5204 0.5138 0.5164 0.5241 0.5203 0.5216 0.5172 0.5178 0.5222 0.5185]
512 [0.5246 0.5297 0.522  0.5229 0.52   0.5242 0.5249 0.5333 0.5338 0.5305]
